# Agregação dos Dados 🎒 🎲

## Explorando as estatísticas brasileiras de comércio exterior

<pre>Seguiremos investigando as bases de dados de exportações e importações brasileiras.</pre>

<pre>Execute as 4 (quatro) células abaixo para criar os 2 (dois) arquivos <b>CSV</b> e carregá-los em DataFrames com a seguinte correspondência:
    - <b>df_sh</b>: Capítulos (dois dígitos) do Sistema Harmonizado, classificação de mercadorias muito usada no comércio internacional; bem como
    - <b>df_pais</b>: Códigos de três dígitos de país usados nas bases de dados da Secretaria de Comércio Exterior.</pre>

In [1]:
!pip install --upgrade pandas --quiet

In [2]:
import requests
import os
import pandas as pd

# desabilitar a verificação de certificado de segurança para fins didáticos
# é possível, considerando que estamos tratando apenas de dados públicos

requests.packages.urllib3.disable_warnings(requests.packages.urllib3.exceptions.InsecureRequestWarning)

# função criada para baixar o arquivo para a pasta data
def download_file(url):
    response = requests.get(url, verify = False) 
    
    if response.status_code == 200:
        content = response.content

        if not os.path.exists('data'): os.makedirs('data')

        with open(f'data/{os.path.basename(url)}', mode='wb') as csvfile:
            csvfile.write(content)

In [3]:
# baixar o arquivo com o mapeamento do código para o nome de cada país
download_file("https://balanca.economia.gov.br/balanca/bd/tabelas/PAIS.csv")

# abrir o arquivo com o mapeamento do código para o nome de cada país
df_pais = pd.read_csv('data/PAIS.csv', encoding='iso-8859-1', header=0, sep=";", dtype = {'CO_PAIS': 'str'})
df_pais = df_pais.set_index("CO_PAIS")

In [4]:
# baixar o arquivo com o mapeamento do código para a descrição de cada NCM
download_file("https://balanca.economia.gov.br/balanca/bd/tabelas/NCM_SH.csv")

# abrir o arquivo com o mapeamento do código para a descrição de cada NCM
df_sh = pd.read_csv('data/NCM_SH.csv', encoding='iso-8859-1', header=0, sep=";", dtype = {'CO_SH2': 'str'})[["CO_SH2", "NO_SH2_POR"]]
df_sh = df_sh.drop_duplicates()
df_sh = df_sh.set_index("CO_SH2")

<pre>A partir do DataFrame <b>df_pais</b>, apenas hoje, nos interessamos pelos códigos -- designados como índice do DataFrame -- e os nomes dos países em português -- presentes na coluna <b>NO_PAIS</b>. Portanto, selecione apenas a coluna/variável <b>NO_PAIS</b> do DataFrame <b>df_pais</b>.</pre>

<details>
  <summary>Resposta</summary>

<br/>

```python
df_pais = df_pais[["NO_PAIS"]]
```

<br/>

</details>

<pre>Carregaremos, a seguir, os dados de exportação do último ano, com valores em dólares americanos -- presentes na coluna variável <b>VL_FOB</b> -- agregados por país de destino e produto no nível do subitem (8 dígitos) da Nomenclatura Comum do Mercosul (NCM).</pre>

In [5]:
import datetime

ultimo_ano = datetime.datetime.now().year - 1

download_file(f"https://balanca.economia.gov.br/balanca/bd/comexstat-bd/ncm/EXP_{ultimo_ano}.csv")

df_exp_ultimo_ano = pd.read_csv(f"data/EXP_{ultimo_ano}.csv", encoding='iso-8859-1', 
                                header=0, sep=";", dtype = {'CO_NCM': 'str', 'CO_PAIS': 'str'})
df_exp_ultimo_ano = df_exp_ultimo_ano[["CO_ANO", "CO_NCM", "CO_PAIS", "VL_FOB"]]

<pre>Agora, vamos transportar os códigos do nível mais desagregado (subitem <b>NCM</b>, de oito dígitos) para o mais agregado (<b>Capítulo SH</b>, de dois dígitos), sabendo que o <b>Capítulo SH</b> corresponde aos dois primeiros dígitos do código <b>NCM</b>.</pre>

<pre>Portanto, crie a coluna/variável <b>CO_SH2</b> no DataFrame <b>df_exp_ultimo_ano</b>, a partir da captura dos dois primeiros dígitos de cada célula da coluna/variável <b>CO_NCM</b>.

👉 dica: veja a documentação do método <a href="https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.map.html">map</a> de um DataFrame.

<details>
  <summary>Resposta</summary>

<br/>

```python
df_exp_ultimo_ano["CO_SH2"] = df_exp_ultimo_ano["CO_NCM"].map(lambda x: x[:2])
```

<br/>

</details>

### Aspectos de GroupBy 🎒 

<pre>Agora, identifique os produtos mais exportados no ano passado, a partir da soma dos valores de exportação do último ano (<b>VL_FOB</b>) por Capítulo SH (<b>CO_SH2</b>), no DataFrame <b>df_exp_ultimo_ano</b>. Atribua o resultado à variável <b>df_principais_produtos</b>.</pre>

👉 dica: veja a documentação do método <a href="https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html">groupby</a> de um DataFrame, bem como do método <a href="https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.agg.html">agg</a>.

<details>
  <summary>Resposta</summary>

<br/>

```python
df_principais_produtos = df_exp_ultimo_ano.groupby("CO_SH2").agg({'VL_FOB': 'sum'})
```

<br/>

</details>

<pre>Inclua os nomes correspondentes aos códigos de produto <b>CO_SH2</b> no mesmo DataFrame <b>df_principais_produtos</b>, realizando uma junção entre o DataFrame <b>df_principais_produtos</b> e o DataFrame <b>df_sh</b>.</pre>

<details>
  <summary>Resposta</summary>

<br/>

```python
df_principais_produtos = df_principais_produtos.join(df_sh).sort_values('VL_FOB', ascending = False)
```

<br/>

</details>

<pre>Agora, ordene o DataFrame <b>df_principais_produtos</b> pelo valor exportado (<b>VL_FOB</b>), de forma decrescente.</pre>

<details>
  <summary>Resposta</summary>

<br/>

```python
df_principais_produtos = df_principais_produtos.sort_values('VL_FOB', ascending = False)
```

<br/>

</details>

<pre>Inspecione o resultado das operações anteriores no DataFrame <b>df_principais_produtos</b> e veja os produtos que o Brasil mais exportou no último ano.</pre>

In [6]:
# descomente a linha abaixo e execute a célula
# df_principais_produtos.head()

<pre>Agora, de forma semelhante às operações anteriores, execute uma sequência de comandos no DataFrame <b>df_exp_ultimo_ano</b> para encontrar os principais destinos/países das exportações brasileiras, a partir da soma dos valores de exportação do último ano (<b>VL_FOB</b>), por País (<b>CO_PAIS</b>). Atribua o resultado à variável <b>df_principais_destinos</b>.</pre>

👉 dica: diferentemente do caso anterior, agora o agrupamento ocorre pelo código do país (<b>CO_PAIS</b>) e a junção ocorre com o DataFrame <b>df_pais</b>.

<details>
  <summary>Resposta</summary>

<br/>

```python
df_principais_destinos = df_exp_ultimo_ano.groupby("CO_PAIS").agg({'VL_FOB': 'sum'})
df_principais_destinos = df_principais_destinos.join(df_pais)
df_principais_destinos = df_principais_destinos.sort_values('VL_FOB', ascending = False)
```

<br/>

</details>

<pre>Inspecione o resultado das operações anteriores no DataFrame <b>df_principais_destinos</b> e veja os países para os quais o Brasil mais exportou no último ano.</pre>

In [7]:
# descomente a linha abaixo e execute a célula
# df_principais_destinos.head()

### Desafio ⚡️

#### Dividir ➗ para Conquistar ➕ (apply)

<pre>Agora, vamos detalhar os principais produtos exportados por destino.</pre>

<pre>Execute a célula abaixo para guardar essa informação no DataFrame <b>df_principais_produtos_destinos</b>.</pre>

In [8]:
df_exp_ultimo_ano["CO_SH2"] = df_exp_ultimo_ano["CO_NCM"].map(lambda x: x[:2])
df_principais_produtos_destinos = df_exp_ultimo_ano.groupby(["CO_PAIS", "CO_SH2"]).agg({'VL_FOB': 'sum'})
df_principais_produtos_destinos = df_principais_produtos_destinos.join(df_pais)
df_principais_produtos_destinos = df_principais_produtos_destinos.join(df_sh)

<pre>Agora, crie um DataFrame que liste, para cada país de destino, apenas os <b>três</b> principais produtos exportados. 

<details>
  <summary>Resposta</summary>

<br/>

```python
df_principais_produtos_destinos = df_principais_produtos_destinos.groupby("NO_PAIS").apply(lambda x: x.sort_values(by="VL_FOB", ascending=False).head(3), include_groups=False)
```

<br/>

</details>

<pre>No ano de 2023, para os Estados Unidos, o resultado da operação acima resultaria a tabela abaixo.</pre>

<table>
    <tr>
        <th colspan=2></th>
        <th>VL_FOB</th>
        <th>NO_SH2_POR</th>
    </tr>
    <tr>
        <th>CO_PAIS</th>
        <th>CO_SH2</th>
        <th colspan=2></th>
    </tr>
    <tr>
        <th rowspan=3>249</th>
        <th>72</th>
        <td>6883892240</td>
        <td>Ferro fundido, ferro e aço</td>
    </tr>
    <tr>
        <th>27</th>
        <td>6043098648</td>
        <td>Combustíveis minerais, óleos minerais e produt...</td>
    </tr>
    <tr>
        <th>84</th>
        <td>3944865403</td>
        <td>Reatores nucleares, caldeiras, máquinas, apare...</td>
    </tr>
</table>

<pre>Por fim, execute a célula abaixo para obter os três principais produtos exportados para os Estados Unidos, a partir do Brasil.</pre>

In [9]:
# descomente a linha abaixo e execute a célula
# df_principais_produtos_destinos.loc["Estados Unidos"]

## Explorando as estatísticas de gênero do Banco Mundial

<pre>Seguiremos investigando as estatísticas relacionadas a gênero na bases de dados do Banco Mundial.</pre>

<pre>Vamos carregar novamente as séries de dados sobre participação na força de trabalho da população masculina adulta.</pre>

<pre>Execute as 5 (cinco) células abaixo para criar os 3 (três) arquivos <b>CSV</b> e carregá-los em DataFrames com a seguinte correspondência:
    - <b>df_labor_foce_country</b>: Taxas de participação feminina e masculina, e a diferença (gap) entre as participações, por país, para o ano de 2019;
    - <b>df_country_regions</b>: Código e nome de cada país, e o código de sua região geográfica; bem como
    - <b>df_region_names</b>: Código e nome de cada região geográfica.</pre>

In [10]:
!pip install --upgrade wbgapi --quiet

In [11]:
import wbgapi as wb
import pandas as pd

In [12]:
economy_info = wb.economy.info()

country_regions_dict = {"country_code": [country.get("id") for country in economy_info.items],
                        "country_name": [country.get("value") for country in economy_info.items],
                        "region_code": [country.get("region") for country in economy_info.items]}

df_country_regions = pd.DataFrame(country_regions_dict).query("region_code != ''")

df_country_regions.to_csv("data/country_regions.csv", index = False)

df_country_regions = pd.read_csv("data/country_regions.csv").set_index(["region_code", "country_code"])

In [13]:
region_info = wb.region.info()

regions_dict = {'region_code': [region.get("code") for region in region_info.items],
               'region_name': [region.get("name") for region in region_info.items]}

df_region_names = pd.DataFrame(regions_dict)

df_region_names.to_csv("data/region_names.csv", index = False)

df_region_names = pd.read_csv("data/region_names.csv").set_index("region_code")

In [14]:
economy_info = wb.economy.info()

country_codes = [pais.get('id') 
                 for pais in economy_info.items
                 if not pais.get('aggregate')]

female_labor_force = wb.data.DataFrame('SL.TLF.ACTI.FE.ZS', time = 2019).squeeze()
male_labor_force = wb.data.DataFrame('SL.TLF.ACTI.MA.ZS', time = 2019).squeeze()

female_labor_force_country = female_labor_force[country_codes]
male_labor_force_country = male_labor_force[country_codes]

df_labor_force_country = pd.DataFrame({"female_rate": female_labor_force_country,
                                       "male_rate": male_labor_force_country,
                                       "rate_gap": male_labor_force_country - female_labor_force_country})

df_labor_force_country = df_labor_force_country.dropna(axis = 0)

df_labor_force_country.to_csv("data/labor_force_country.csv", index=True)

df_labor_force_country = pd.read_csv("data/labor_force_country.csv").rename(columns={"economy": "country_code"}).set_index("country_code")

<pre>Inspecione o DataFrame <b>df_labor_force_country</b> e veja as taxas de participação feminina e masculina, e a diferença (gap) entre as participações, por país, para o ano de 2019.</pre>

In [15]:
df_labor_force_country.head()

,female_rate,male_rate,rate_gap
country_code,,,
AFG,19.062,70.659,51.597
AGO,74.079,77.909,3.830
ALB,61.901,77.647,15.746
ARE,55.540,92.277,36.737
ARG,58.254,77.857,19.603


<pre>Agora, inspecione o DataFrame <b>df_country_regions</b> e veja o código e nome de cada país, bem como  o código de sua região geográfica.</pre>

In [16]:
df_country_regions.head()

country_name
region_code country_code             
LCN         ABW                 Aruba
SAS         AFG           Afghanistan
SSF         AGO                Angola
ECS         ALB               Albania
            AND               Andorra

<pre>Por fim, inspecione o DataFrame <b>df_region_names</b> e veja a correspondência entre os códigos e os nomes das regiões.</pre>

In [17]:
df_region_names.head()

,region_name
region_code,
AFE,Africa Eastern and Southern
AFR,Africa
AFW,Africa Western and Central
ARB,Arab World
CAA,Sub-Saharan Africa (IFC classification)


### Desafio ⚡️

#### Dividir ➗ para Conquistar ➕ (apply)

<pre>Você deverá criar um novo DataFrame <b>df_country_regions_names</b> que apresente, para cada região geográfica, o país com maior <b>gap</b> nas taxas de participação na força de trabalho.</pre>

<details>
  <summary>Resposta</summary>

<br/>

```python
df_country_regions_names = df_country_regions.join(df_region_names, how = "left")
```

<br/>

</details>

<pre>Por fim, crie um DataFrame <b>biggest_gap_region</b> que contenha somente as colunas de nome da região, nome do país (que apresentou o maior <b>gap</b>), suas taxas de participação feminina e masculina, e o <b>gap</b> propriamente dito.

<details>
  <summary>Resposta</summary>

<br/>

```python
biggest_gap_region = labor_force_country_df.join(country_regions_names_df, how = "left")
biggest_gap_region = biggest_gap_region.groupby("region_name").apply(lambda df: df.sort_values('rate_gap', ascending = False).head(1), include_groups=False)
biggest_gap_region
```

<br/>

</details>